# 04 — Karnataka Operational STAC Inference & Backend Export

Searches Microsoft Planetary Computer STAC for `sentinel-1-rtc`, `sentinel-2-l2a`, `jrc-gsw`, and `esa-worldcover` over the Bengaluru/Karnataka AOI, converts S1 linear power to dB, runs sliding-window U-Net inference, computes NDVI/BSI land-change polygons, and exports outputs (`sample: false`) to `backend/data/`.

In [ ]:
import json
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve().parent))

import numpy as np
import geopandas as gpd
from ml.config import AOI_BBOX, BACKEND_DATA_DIR, BACKEND_IMAGES_DIR, SEN1FLOODS11_DIR
from ml.src.stac_fetch import export_scenes_metadata, search_sentinel1_rtc
from ml.src.preprocess import plot_db_histogram_comparison, read_and_align_s1_rtc_pair
from ml.src.infer import sliding_window_flood_inference
from ml.src.postprocess import clean_flood_mask, polygonize_mask
from ml.src.export_results import export_pipeline_outputs, save_gray_preview_png

scenes_meta = export_scenes_metadata()
s1_res = search_sentinel1_rtc()
if not s1_res['items']:
    raise RuntimeError(s1_res['warning'])

latest_s1 = s1_res['items'][0]
s1_stack, profile = read_and_align_s1_rtc_pair(
    latest_s1.assets['vv'].href,
    latest_s1.assets['vh'].href,
    AOI_BBOX,
)
save_gray_preview_png(s1_stack[1], BACKEND_IMAGES_DIR / 'sar_after.png')
sf_chips = list((SEN1FLOODS11_DIR / 'S1Hand').glob('*.tif'))
if sf_chips:
    plot_db_histogram_comparison(sf_chips, s1_stack)

raw_mask, prob_map = sliding_window_flood_inference(s1_stack)
cleaned_mask = clean_flood_mask(raw_mask)
flood_gdf = polygonize_mask(cleaned_mask, profile['transform'], prob_map=prob_map)
model_card = json.loads((BACKEND_DATA_DIR / 'model_card.json').read_text())
print('Flood polygons extracted (Model prediction, unvalidated):', len(flood_gdf))